## Chapter 05: Embedding LLMs within Your Applications

### Getting started with LangChain

#### 1. Models and prompts

In [58]:
import environment
import importlib

importlib.reload(environment)

from langchain_deepseek import ChatDeepSeek

In [ ]:
# LangChain model integrations
llm = ChatDeepSeek(
  model="deepseek-chat",
  api_key=config.DEEPSEEK_API_KEY,
  temperature=0,
  max_tokens=None,
  timeout=None,
  max_retries=2,
)

In [7]:
print(
  llm.invoke("Tell me a joke").content
)

Why don't scientists trust atoms?

Because they make up everything!


In [8]:
# Prompt templates
from langchain_core.prompts import PromptTemplate

In [10]:
# Instantiation
template = """
Sentence: {sentence}

Translation in {language}:
"""

prompt = PromptTemplate.from_template(template)

print(
  prompt.format(
    sentence = "The cat is on the table.",
    language = "spanish"
	)
)


Sentence: The cat is on the table.

Translation in spanish:



In [ ]:
# Example selector
from langchain_core.example_selectors import BaseExampleSelector

In [49]:
example = BaseExampleSelector.add_example(
  self=BaseExampleSelector,
  example={
    "prompt": "What is your name?",
    "completion": "I go by Sally.",
	}
)

In [32]:
print(example)

None


#### 2. Data connections

- Document loaders

In [ ]:
from langchain_community.document_loaders import CSVLoader

In [36]:
loader = CSVLoader(file_path="./data/sample.csv")
data = loader.load()

In [37]:
print(data)

[Document(metadata={'source': './data/sample.csv', 'row': 0}, page_content='Name: John\nAge: 25\nCity: New York'), Document(metadata={'source': './data/sample.csv', 'row': 1}, page_content='Name: Emily\nAge: 28\nCity: Los Angeles'), Document(metadata={'source': './data/sample.csv', 'row': 2}, page_content='Name: Michael\nAge: 22\nCity: Chicago')]


- Document transformers

In [ ]:
with open("./data/mountain.txt") as f:
  mountain = f.read()

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [41]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=100, 				# number of characters for each chunk
  chunk_overlap=20, 			# number of characters overlapping between a preceding and following chunk
  length_function=len, 		# function used to measure the number of characters
)

texts = text_splitter.create_documents([mountain])

In [42]:
print(texts[0])
print(texts[1])
print(texts[2])

page_content='Amidst the serene landscape, towering mountains stand as majestic guardians of nature's beauty.'
page_content='The crisp mountain air carries whispers of tranquility, while the rustling leaves compose a'
page_content='leaves compose a symphony of wilderness.'


- Text embedding models

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [146]:
embedding_model = GoogleGenerativeAIEmbeddings(
  model="gemini-embedding-001",
  api_key=environment.GEMINI_API_KEY
)

# Batch (embed multiple strings at once for a processing speedup)
embeddings = embedding_model.embed_documents(
  [
    "Good morning!",
    "Oh, hello!",
    "I want to report an accident",
    "Sorry to hear that. May I ask your name?",
    "Sure. Mario Rossi."
	]
)

In [148]:
print("Embed documents:")
print(
  f"Number of vectors: {len(embeddings)}; Dimension of each vector: {len(embeddings[0])}"
)
print("-" * 50)

embedded_query = embedding_model.embed_query(
  "What was the name mentioned in the conversation?"
)

print("Embed query:")
print(
  f"Dimension of the vector: {len(embedded_query)}"
)
print(
  f"Sample of the first 5 elements of the vector: {embedded_query[:5]}"
)

Embed documents:
Number of vectors: 5; Dimension of each vector: 3072
--------------------------------------------------
Embed query:
Dimension of the vector: 3072
Sample of the first 5 elements of the vector: [-0.030619625, 0.0032838325, 0.015690394, -0.0870063, -0.008714505]


- Vector stores (Vector DBs)

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [151]:
# Load the document, split it into chunks, embed each chunk and load it into the vector store
raw_documents = TextLoader("./data/dialogue.txt").load()
text_splitter = CharacterTextSplitter(
  chunk_size=50,
  chunk_overlap=0,
  separator="\n"
)
documents = text_splitter.split_documents(raw_documents)
embedding_model = GoogleGenerativeAIEmbeddings(
  model="gemini-embedding-001",
  api_key=environment.GEMINI_API_KEY
)
vector_store = InMemoryVectorStore(embedding=embedding_model)

vector_store.add_documents(documents=documents)

['57e6a59e-d069-4e17-960a-0f4e39a81b45',
 '5023ed54-0605-45b2-82d9-db9dcbd90ee0',
 'be38f655-3501-4ac0-815b-20b58e63c798',
 'ed8bb6f2-9aa1-4b0e-a454-b14b1ea84d4c']

In [156]:
query = "What is the reason for calling?"
search_query = vector_store.similarity_search(query)

print(search_query[0].page_content)

Sorry to hear that. May I ask your name?
